# Eye Movement-Based Schizophrenia Recognition — Full Pipeline

| Cell | Mục đích |
|---|---|
| 1 | Mount Drive + cd vào project |
| 2 | 🔴 XÓA kết quả cũ (bỏ comment khi cần reset) |
| 3 | Install thư viện còn thiếu |
| 4 | Tier 1 — Preprocessing |
| 5 | Tier 2 — Feature Engineering |
| 6 | Tier 3 — Tabular (XGBoost) |
| 7 | Tier 4A — ResNet50 extraction |
| 8 | Tier 4B — Build graphs |
| 9 | Tier 4C — GNN-CEFAM training |
| 10 | Tier 4D — BiCA-HS training |
| 11 | Tier 5 — Meta-Learner |
| 12 | Tổng hợp kết quả |

In [1]:
from google.colab import drive
drive.mount('/content/drive')

%cd "/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition


In [2]:
# 🔴 XÓA KẾT QUẢ CŨ — bỏ comment từng dòng tùy mức độ reset

# Xóa checkpoint + OOF Tier 4 (BẮT BUỘC sau khi fix C-1, C-2, C-3)
# !rm -rf results/bica/ results/cefam/ results/stgnn/ results/tier5/
# !rm -rf "Bidirectional Cross-Attention Hybrid Stream/results/checkpoints/"

# Xóa Tier 3
# !rm -rf results/baselines/

# Xóa graphs (rebuild từ đầu)
# !rm -rf data/processed/graphs/

# Xóa toàn bộ (chạy lại từ raw data)
!rm -rf results/ data/processed/ data/external/

In [3]:
# Colab đã có sẵn torch/sklearn/pandas — chỉ install thêm cái còn thiếu
!pip install torch-geometric
!pip install pyyaml omegaconf optuna catboost pyarrow openpyxl xgboost lightgbm shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 24.9 MB/s eta 0:00:00


In [4]:
# Tier 1: Tạo category map + tiền xử lý dữ liệu thô
!python -m src.utils.generate_category_map
!python -m src.tier1_preprocessing.preprocess

Successfully generated category mapping for 100 images at data/metadata/stimulus_categories.csv
--- Tier 1: Loading raw data ---
Loading Fixations: 100% 160/160 [00:18<00:00,  8.66it/s]
Loading Fixations: 100% 48/48 [00:36<00:00,  1.31it/s]
Total raw fixations loaded: 293740
Spatial boundary filter: removed 5037 out of 293740 fixations (1.71%)
Temporal duration filter: removed 7666 out of 288703 fixations (2.66%)
Successfully saved 281037 fixations to data/processed/clean_fixations.parquet
--- Tier 1 Preprocessing Completed Successfully ---


In [5]:
# Tier 2: Trích xuất đặc trưng stimulus-level + subject-level delta
!python -m src.tier2_features.stimulus_features
!python -m src.tier2_features.subject_aggregator

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Extracting features per trial...
100% 20695/20695 [00:23<00:00, 885.20it/s]
Extracted features for 20695 trials. Saved to data/processed/features_stimulus_level.csv
Loading stimulus-level features from data/processed/features_stimulus_level.csv...
Loading category mapping from data/metadata/stimulus_categories.csv...
Computing mean feature values per subject, per category...
Computing contextual delta features...
Aggregated subject-level features for 208 subjects. Saved to data/processed/features_subject_level.csv


In [6]:
# Tier 3: Tabular baseline (XGBoost)
!python -m src.tier3_tabular.tabular_models --model xgboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [09:21:29] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iter

In [2]:
!python -m src.tier3_tabular.tabular_models --model lightgbm

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.7615
  Stimulus-Level Auc: 0.8385
  Stimulus-Level F1: 0.7692
  Stimulus-Level Precision: 0.7423
  St

In [3]:
!python -m src.tier3_tabular.tabular_models --model catboost

Loading stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
Loading CV splits from EMS/Train_Valid.xlsx...
Number of features (stimulus + delta): 135
Training on 160 subjects (15907 trials)...

--- Fold 0 ---

--- Fold 1 ---

--- Fold 2 ---

--- Fold 3 ---

--- Evaluation on Stimulus Level ---
  Stimulus-Level Accuracy: 0.8114
  Stimulus-Level Auc: 0.8971
  Stimulus-Level F1: 0.8057
  Stimulus-Level Precision: 0.8272
  St

In [7]:
# Tier 4A: Trích xuất ResNet50 visual features (2048-dim)
# Output: data/external/feature_dict_ResNet50.npy
!python -m src.utils.extract_resnet_features

 VISUAL FEATURE EXTRACTION (ResNet50 Baseline)
Found 100 stimulus images in EMS/Images.
Loading pre-trained ResNet50 on device: cuda...
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 242MB/s]
Extracting feature maps for all stimulus images...
Extracting image features: 100% 100/100 [01:12<00:00,  1.38it/s]
Mapping visual features to subject fixations...
Mapping to trials: 100% 16716/16716 [00:56<00:00, 295.45it/s]

Successfully extracted ResNet50 features. Saved to data/external/feature_dict_ResNet50.npy
Total trials mapped: 16716
Feature vector dimension: 2048


In [8]:
# Tier 4B: Xây dựng đồ thị PyG từ fixation data + ResNet50 features
# Output: data/processed/graphs/graphs.pt
!python -m src.tier4_advanced.graph_builder

Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Global pupil stats (minor leakage — see NOTE above): Mean=1218.29, Std=614.17
Loading ResNet50 features from data/external/feature_dict_ResNet50.npy...
Detected visual feature dimension from dataset: 2048
Building spatiotemporal graphs...
100% 16716/16716 [00:37<00:00, 448.59it/s]
Successfully constructed and saved 16716 graphs at data/processed/graphs/graphs.pt


In [9]:
# Tier 4C: Huấn luyện GNN-CEFAM (4-fold GroupKFold)
# Output: results/cefam/cefam_oof_subject_preds.csv
!python scripts/train_tier4.py

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
  [Delta] Loaded 120 delta features from data/processed/features_subject_level.csv
Handcrafted feature dimension: 135 (15 stimulus + 120 delta)
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0448 TrAUC=0.9410 | VaLoss=0.0864 VaTrialAUC=0.8671 VaSubjAUC=0.8864
Ep 005/200 | TrLoss=0.0087 TrAUC=0.9977 | VaLoss=0.0919 VaTrialAUC

In [4]:
!python scripts/train_tier4.py --config configs/stgnn_config.yaml

Using device: cuda
Loaded clean fixations. Global pupil mean: 1218.2891, std: 614.1672
Loading spatiotemporal graphs from data/processed/graphs/graphs.pt...
Total train/valid graphs: 15598
Total test graphs: 1118

==================== Training Fold 0 ====================
Fold 0 training pupil stats: Mean=1217.3096, Std=610.6889
Train graphs: 11686, Val graphs: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier4_advanced/gnn_stream.py:58: UserWarning: 'nn.glob.GlobalAttention' is deprecated, use 'nn.aggr.AttentionalAggregation' instead
  self.pool = GlobalAttention(gate_nn)
Ep 001/200 | TrLoss=0.0630 TrAUC=0.8091 | VaLoss=0.0652 VaTrialAUC=0.8237 VaSubjAUC=0.8914
Ep 005/200 | TrLoss=0.0446 TrAUC=0.8940 | VaLoss=0.0688 VaTrialAUC=0.8100 VaSubjAUC=0.8561
Ep 010/200 | TrLoss=0.0418 TrAUC=0.9076 | VaLoss=0.0579 VaTrialAUC=0.8372 VaSubjAUC=0.9040
Ep 015/200 | TrLoss=0.0382 TrAUC=0.9236 | VaLoss=0.0667 VaTrialAUC=0.8419 VaSubjAUC=0.8889
Ep 020/200 | TrLoss=0.036

In [9]:
!python scripts/calibrate_threshold.py --preds results/stgnn/stgnn_subject_val_predictions.csv

Loading predictions from results/stgnn/stgnn_subject_val_predictions.csv...
Overall Subject-Level AUC: 0.9309

--- Metrics at Default Threshold (0.5000) ---
  Accuracy: 0.7812
  F1-Score: 0.7368
  Precision: 0.9245
  Recall: 0.6125

--- Optimized Metrics at Calibrated Threshold (0.4126) ---
  Accuracy: 0.8625
  F1-Score: 0.8675
  Precision: 0.8372
  Recall (Sensitivity): 0.9000
  Specificity: 0.8250


In [10]:
# Tier 4D: Huấn luyện BiCA-HS (4-fold GroupKFold)
# PYTHONPATH=. bắt buộc để import src.*
# Output: results/bica/bica_subject_val_predictions.csv
!PYTHONPATH=. python "Bidirectional Cross-Attention Hybrid Stream/scripts/train_bica.py" --config configs/bica_config.yaml

Using device: cuda
Loading preprocessed fixations from data/processed/clean_fixations.parquet...
Loading flat stimulus-level features from data/processed/features_stimulus_level.csv...
Total train/valid trials: 15598
Total test trials: 1118

==================== Training Fold 0 ====================
Train trials: 11686, Val trials: 3912
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/models/bica/model.py:52: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Epoch 001/150 | Train Loss: 0.0597 | Train AUC: 0.8978 | Val Loss: 0.0889 | Val Trial AUC: 0.7859 | Val Subject AUC: 0.8106
Epoch 005/150 | Train Loss: 0.0436 | Train AUC: 0.9279 | Val Loss: 0.0967 | Val Trial AUC: 0.8252 | Val Subject AUC: 0.8712
Epoch 010/150 | Train Loss: 0.0418 | Train AUC: 0.9325 | Val Loss: 0.0650 | Val Trial AUC: 0.8576 | Val Sub

In [11]:
# Tier 5: Meta-Learner kết hợp Tier 3 + Tier 4
# Mặc định dùng XGBoost OOF + CEFAM OOF
!python scripts/run_tier5.py --plot --calibrate

2026-06-27 10:44:46.404727: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 10:44:46.474732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.003, 0.989]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9405  ACC: 0.8

In [12]:
# Tier 5 (phiên bản dùng BiCA-HS OOF thay vì CEFAM)
!python scripts/run_tier5.py \
    --tier4-oof results/bica/bica_subject_val_predictions.csv \
    --plot --calibrate

2026-06-27 10:45:09.708453: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 10:45:09.782291: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.007, 0.889]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8702  ACC: 0.7812  Brier: 0.1779  BSS: 0.2886  ECE: 0.1710

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9522  ACC: 0.9

In [8]:
!python scripts/run_tier5.py \
    --tier3-oof results/baselines/catboost_oof_subject_preds.csv \
    --tier4-oof results/bica/bica_subject_val_predictions.csv \
    --plot --calibrate

2026-06-27 12:30:47.565138: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-27 12:30:47.635638: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
 TIER 5 META-LEARNER
[Tier5 OOF] Loaded 160 subjects | P_tab range [0.000, 1.000] | P_bica range [0.007, 0.889]
[Tier5] Test prediction files not found; skipping test inference.

--- Baseline: Tier 3 alone (XGBoost/LGB/CatBoost) ---
  AUC: 0.8981  ACC: 0.8125  Brier: 0.1480  BSS: 0.4082  ECE: 0.1508

--- Baseline: Tier 4 alone (BiCA-HS / CEFAM) ---
  AUC: 0.9522  ACC: 0.9

In [13]:
# Tổng hợp kết quả
import json, os
import pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

def show(path, name):
    if not os.path.exists(path):
        return
    df = pd.read_csv(path)
    if 'Label' not in df.columns or 'Pred_Proba_Subject' not in df.columns:
        return
    y, p = df['Label'].values, df['Pred_Proba_Subject'].values
    print(f"{name:<35} AUC={roc_auc_score(y,p):.4f}  ACC={accuracy_score(y,(p>=.5).astype(int)):.4f}  F1={f1_score(y,(p>=.5).astype(int)):.4f}")

print("=" * 70)
show("results/baselines/xgboost_oof_subject_preds.csv",  "Tier 3 XGBoost")
show("results/cefam/cefam_oof_subject_preds.csv",        "Tier 4 GNN-CEFAM")
show("results/bica/bica_subject_val_predictions.csv",    "Tier 4 BiCA-HS")

t5 = "results/tier5/tier5_results_summary.json"
if os.path.exists(t5):
    d = json.load(open(t5))
    if "best_model" in d:
        b = d["best_model"]
        print(f"{'Tier 5 Meta-Learner':<35} AUC={b.get('auc',0):.4f}  ACC={b.get('acc',0):.4f}  F1={b.get('f1',0):.4f}")
print("=" * 70)
print("SOTA (MSNet, IEEE TNNLS 2025):      AUC=0.8854  ACC=0.8125")

Tier 3 XGBoost                      AUC=0.8702  ACC=0.7812  F1=0.7853
Tier 4 GNN-CEFAM                    AUC=0.9405  ACC=0.8688  F1=0.8679
Tier 4 BiCA-HS                      AUC=0.9522  ACC=0.9062  F1=0.9123
SOTA (MSNet, IEEE TNNLS 2025):      AUC=0.8854  ACC=0.8125


In [5]:
!rm -rf experiments/ablation/results/ experiments/ablation/figures/

!python experiments/ablation/run_ablation_analysis.py

!python experiments/ablation/plot_ablation.py

ABLATION STUDY - Eye Movement-Based Schizophrenia Recognition

Loading model predictions...
Loaded 4 models: ['GNN+CEFAM (Full Hybrid)', 'ST-GNN (GNN Only)', 'BiCA-HS (Transformer)', 'XGBoost (Tabular Only)']

F1: FULL MODEL COMPARISON (Main Ablation Table)

--- GNN+CEFAM (Full Hybrid) ---
  AUC-ROC:  0.9405
  ACC @0.5: 0.8688
  F1  @0.5: 0.8679
  ACC @opt: 0.8750 (th=0.3026)
  F1  @opt: 0.8810
  Sens@opt: 0.9250
  Spec@opt: 0.8250

--- ST-GNN (GNN Only) ---
  AUC-ROC:  0.9309
  ACC @0.5: 0.7812
  F1  @0.5: 0.7368
  ACC @opt: 0.8625 (th=0.4126)
  F1  @opt: 0.8675
  Sens@opt: 0.9000
  Spec@opt: 0.8250

--- BiCA-HS (Transformer) ---
  AUC-ROC:  0.9522
  ACC @0.5: 0.9062
  F1  @0.5: 0.9123
  ACC @opt: 0.9062 (th=0.4990)
  F1  @opt: 0.9123
  Sens@opt: 0.9750
  Spec@opt: 0.8375

--- XGBoost (Tabular Only) ---
  AUC-ROC:  0.8702
  ACC @0.5: 0.7812
  F1  @0.5: 0.7853
  ACC @opt: 0.8063 (th=0.5953)
  F1  @opt: 0.8050
  Sens@opt: 0.8000
  Spec@opt: 0.8125

F2: COMPONENT CONTRIBUTION ANALYSIS

 

In [6]:
!python scratch_delong_test.py

 DELONG SIGNIFICANCE TEST: BiCA-HS vs Tier 5 Ensembles
Number of subjects : 160
Tier 4 (BiCA-HS) AUC : 0.9522
------------------------------------------------------------
 1. CROSS-VALIDATED ENSEMBLE (Generalization performance)
------------------------------------------------------------
Tier 5 (CV Meta) AUC : 0.9437
Z-statistic          : 0.6074
P-value              : 0.543587
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
------------------------------------------------------------
 2. FULL META-LEARNER ENSEMBLE (Final model deployment)
------------------------------------------------------------
Tier 5 (Full Meta) AUC: 0.9597
Z-statistic           : -0.7482
P-value               : 0.454346
Result is NOT statistically significant at alpha=0.05 (p >= 0.05).
The difference in performance could be due to chance.


In [7]:
!python scratch_feature_importance.py

  [Delta] Merged 120 delta/category-mean features from data/processed/features_subject_level.csv
/content/drive/MyDrive/Eye Movement-Based Schizophrenia Recognition/src/tier3_tabular/tabular_models.py:192: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['Category'] = df_merged['Stimulus_ID'].map(category_map)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [12:25:28] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, i